
# The UVJ colour–colour diagram

The UVJ (U−V vs V−J) diagram is a classic method for separating
star-forming from quiescent galaxies. We populate it with four
model tracks: (1) constant star-forming galaxies with varying
dust optical depth, (2) an old quiescent population, (3) a
post-starburst galaxy, and (4) a dusty starburst. The grey box
marks the "quiescent region" from Williams+2009, a visual guide
for identifying passive galaxies.

dust, age, and star formation
history shape a galaxy's position in the UVJ plane — a
workhorse diagnostic for photometric surveys.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")


def _flux(model, params):
    return np.asarray(model.predict_photometry(params))


def _colors_from_flux(flux_u, flux_v, flux_j):
    """Compute U-V and V-J colors from flux."""
    uv = -2.5 * np.log10(flux_u / flux_v)
    vj = -2.5 * np.log10(flux_v / flux_j)
    return uv, vj


# Load filters: Johnson U, V and 2MASS J
obs = tengri.Observation(
    photometry=tengri.Photometry.from_names(["johnson_u", "johnson_v", "2mass_j"])
)

# ============================================================================
# Track 1: Star-forming with dust sweep (constant SFR, tau_V = 0 to 2)
# ============================================================================
sf_model = tengri.SEDModel.build(
    tengri.load_ssp(),
    observation=obs,
    sfh={"type": "const", "*": tengri.FIXED, "log_sfr": 0.5},
    dust={
        "type": "two_component",
        "*": tengri.FIXED,
        "tau_diff": 0.1,
        "tau_bc": 0.0,
        "slope": -0.7,
    },
    redshift=tengri.Fixed(0.05),
)

tau_v_grid = np.linspace(0, 2.0, 15)
sf_uv, sf_vj = np.empty_like(tau_v_grid), np.empty_like(tau_v_grid)
baseline_sf = dict(sf_model.spec.sample(jax.random.PRNGKey(0)))

for i, tau_v in enumerate(tau_v_grid):
    # tau_v is the birth-cloud optical depth; map to tau_bc
    params = {**baseline_sf, "dust_tau_bc": tau_v}
    flux = _flux(sf_model, params)
    sf_uv[i], sf_vj[i] = _colors_from_flux(flux[0], flux[1], flux[2])

# ============================================================================
# Track 2: Old quiescent population (10 Gyr age)
# ============================================================================
quiescent_model = tengri.SEDModel.build(
    tengri.load_ssp(),
    observation=obs,
    sfh={
        "type": "tsnorm",
        "*": tengri.FIXED,
        "peak_lbt_gyr": 10.0,
        "width_gyr": 0.5,
        "log_total_mass": 10.0,
        "skew": 0.0,
        "trunc": 13.0,
    },
    dust={"type": "two_component", "*": tengri.FIXED, "tau_diff": 0.02, "tau_bc": 0.0},
    redshift=tengri.Fixed(0.05),
)

baseline_q = dict(quiescent_model.spec.sample(jax.random.PRNGKey(1)))
flux_q = _flux(quiescent_model, baseline_q)
q_uv, q_vj = _colors_from_flux(flux_q[0], flux_q[1], flux_q[2])

# ============================================================================
# Track 3: Post-starburst (burst 9 Gyr ago, then quiescent for 1 Gyr)
# ============================================================================
psb_model = tengri.SEDModel.build(
    tengri.load_ssp(),
    observation=obs,
    sfh={
        "type": "tsnorm",
        "*": tengri.FIXED,
        "peak_lbt_gyr": 1.0,
        "width_gyr": 0.2,
        "log_total_mass": 10.0,
        "skew": 0.0,
        "trunc": 13.0,
    },
    dust={"type": "two_component", "*": tengri.FIXED, "tau_diff": 0.05, "tau_bc": 0.2},
    redshift=tengri.Fixed(0.05),
)

baseline_psb = dict(psb_model.spec.sample(jax.random.PRNGKey(2)))
flux_psb = _flux(psb_model, baseline_psb)
psb_uv, psb_vj = _colors_from_flux(flux_psb[0], flux_psb[1], flux_psb[2])

# ============================================================================
# Track 4: Dusty starburst (high SFR, high dust)
# ============================================================================
burst_model = tengri.SEDModel.build(
    tengri.load_ssp(),
    observation=obs,
    sfh={"type": "const", "*": tengri.FIXED, "log_sfr": 1.5},
    dust={
        "type": "two_component",
        "*": tengri.FIXED,
        "tau_diff": 0.3,
        "tau_bc": 1.5,
        "slope": -0.7,
    },
    redshift=tengri.Fixed(0.05),
)

baseline_burst = dict(burst_model.spec.sample(jax.random.PRNGKey(3)))
flux_burst = _flux(burst_model, baseline_burst)
burst_uv, burst_vj = _colors_from_flux(flux_burst[0], flux_burst[1], flux_burst[2])

# ============================================================================
# Plot
# ============================================================================
fig, ax = plt.subplots(figsize=(6.4, 6.0))

# Williams+2009 quiescent box
quiescent_box = mpatches.Rectangle(
    (1.3, 1.6),
    0.5,
    0.5,
    linewidth=1.0,
    edgecolor="grey",
    facecolor="lightgrey",
    alpha=0.3,
    zorder=1,
)
ax.add_patch(quiescent_box)

# Track 1: Star-forming dust sweep
ax.plot(sf_vj, sf_uv, "-o", color="#2ecc71", lw=2.0, markersize=4, label="Star-forming", zorder=3)

# Track 2: Quiescent
ax.scatter(q_vj, q_uv, s=150, color="#e74c3c", marker="s", label="Quiescent (10 Gyr)", zorder=4)

# Track 3: Post-starburst
ax.scatter(
    psb_vj, psb_uv, s=150, color="#f39c12", marker="^", label="Post-starburst (1 Gyr)", zorder=4
)

# Track 4: Dusty starburst
ax.scatter(
    burst_vj, burst_uv, s=150, color="#9b59b6", marker="D", label="Dusty starburst", zorder=4
)

ax.set(
    xlabel=r"$V - J$  [AB mag]",
    ylabel=r"$U - V$  [AB mag]",
    xlim=(-0.4, 2.2),
    ylim=(-0.4, 2.6),
)
ax.legend(frameon=False, fontsize=9, loc="upper left")
ax.grid(True, alpha=0.2, linestyle="--", linewidth=0.5)

fig.tight_layout()
plt.savefig("plot_uvj_diagram.png", dpi=150, bbox_inches="tight")